# Tools & Function Calling

## Build a Tool-Using AI Agent

In this lab, you will build a simple Python application that connects an LLM to three tools:

- Calculator
- Search
- External API (Weather)

You will learn how an LLM can decide which tool is useful, how a tool is described with a schema, how Python executes the tool, and how the tool result is returned to the LLM.

## Learning Objectives

By the end of this lab, you should be able to:

1. Explain what a tool is.
2. Understand a tool schema.
3. Define a function that can be used as a tool.
4. Send tool declarations to an LLM.
5. Identify a function call returned by the model.
6. Execute the requested function in Python.
7. Send the tool result back to the LLM.
8. Build a small tool-using agent.

## 1. How the Agent Works

The basic function-calling flow is:

User Request
↓
LLM
↓
Select a Tool
↓
Python Executes the Tool
↓
Tool Result
↓
LLM
↓
Final Answer

Important:

> The LLM does not execute your Python function itself.

The model returns a structured function call containing the function name and arguments. Your Python application executes the function and sends the result back to the model.

## 2. Setup

We will use Google's `google-genai` Python SDK.

We will also use:

- `requests` for the weather API
- `ddgs` for web search

In [ ]:
!pip -q install google-genai ddgs

## 3. Add Your Gemini API Key

Create a Gemini API key from Google AI Studio.

For this notebook, enter the key when prompted.

Do not share your API key in a public notebook.

In [ ]:
import os
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded successfully.")

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

MODEL = "gemini-3.8-flash"

print("Gemini client is ready.")

## 4. What Is a Tool?

A tool is an external capability that an LLM can request when it needs information or an action that should be handled by another system.

Examples:

| Tool | Purpose |
|---|---|
| Calculator | Performs calculations |
| Search | Finds current information |
| Weather API | Gets live weather data |

The LLM decides **when a tool is useful**.

Your Python application is responsible for **executing the tool**.

## 5. Tool 1 — Calculator

First, we will create a normal Python function.

The LLM will later be given a description of this function so it knows when and how to use it.

In [ ]:
import ast
import operator as op

_ALLOWED_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        left = _safe_eval(node.left)
        right = _safe_eval(node.right)

        return _ALLOWED_OPERATORS[type(node.op)](
            left,
            right
        )

    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](
            _safe_eval(node.operand)
        )

    raise ValueError("Unsupported expression")


def calculator(expression: str):
    """Calculate a basic mathematical expression."""

    try:
        tree = ast.parse(expression, mode="eval")

        result = _safe_eval(tree.body)

        return {
            "expression": expression,
            "result": result
        }

    except Exception:
        return {
            "error": "Invalid or unsupported mathematical expression."
        }


print(calculator("125 * 24"))

## 6. Test the Calculator

Run the cell below.

Expected result:

```text
3000

In [ ]:
result = calculator("125 * 24")

print(result)

## 7. Tool Schema

A tool schema tells the LLM:

- the tool name
- what the tool does
- what arguments it needs
- the type of each argument

Our calculator schema will look like this conceptually:

```text
calculator(expression: string)

In [ ]:
calculator_declaration = {
    "type": "function",
    "name": "calculator",
    "description": "Calculates a basic mathematical expression.",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A mathematical expression such as 25 * 18"
            }
        },
        "required": ["expression"]
    }
}

print(calculator_declaration)

## 8. Tool 2 — Search

Now we will create a search tool.

This tool uses the `ddgs` package to search the web and return a small set of results.

The important idea is not the search library itself.

The important idea is that **search is an external capability that the LLM can request through a tool call**.

In [ ]:
from ddgs import DDGS


def web_search(query: str):
    """Search the web and return a few relevant results."""

    try:
        results = DDGS().text(
            query,
            max_results=3
        )

        simplified = []

        for item in results:
            simplified.append({
                "title": item.get("title"),
                "url": item.get("href"),
                "snippet": item.get("body")
            })

        return simplified

    except Exception as e:
        return {
            "error": f"Search failed: {e}"
        }


print(web_search("What is LangGraph?"))

## 9. Search Tool Schema

In [ ]:
search_declaration = {
    "type": "function",
    "name": "web_search",
    "description": "Searches the web for current or specific information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query."
            }
        },
        "required": ["query"]
    }
}

print(search_declaration)

## 10. Tool 3 — External API

For the third tool, we will use the Open-Meteo API.

It does not require an API key.

Our Python function will:

1. Convert a city name into coordinates.
2. Request current weather data.
3. Return the weather information.

In [ ]:
import requests


def get_weather(city: str):
    """Get current weather for a city using the Open-Meteo API."""

    try:
        # Find the city
        geo_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=10
        )

        geo_response.raise_for_status()

        geo_data = geo_response.json()

        if not geo_data.get("results"):
            return {
                "error": f"City not found: {city}"
            }

        location = geo_data["results"][0]

        latitude = location["latitude"]
        longitude = location["longitude"]
        city_name = location["name"]

        # Get current weather
        weather_response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,wind_speed_10m",
                "timezone": "auto"
            },
            timeout=10
        )

        weather_response.raise_for_status()

        weather_data = weather_response.json()

        current = weather_data["current"]

        return {
            "city": city_name,
            "temperature_c": current["temperature_2m"],
            "wind_speed_kmh": current["wind_speed_10m"]
        }

    except Exception as e:
        return {
            "error": f"Weather request failed: {e}"
        }


print(get_weather("Islamabad"))

## 11. Weather Tool Schema

In [ ]:
weather_declaration = {
    "type": "function",
    "name": "get_weather",
    "description": "Gets the current weather for a city using an external weather API.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city."
            }
        },
        "required": ["city"]
    }
}

print(weather_declaration)

## 12. Put the Tools Together

Now we have three tools:

```text
calculator(expression)
web_search(query)
get_weather(city)

In [ ]:

TOOL_DECLARATIONS = [
    calculator_declaration,
    search_declaration,
    weather_declaration
]

print("Available tools:")

for tool in TOOL_DECLARATIONS:
    print("-", tool["name"])

## 13. Create the Tool Map

The LLM returns a tool name as part of the function call.

Our Python application needs to know which Python function belongs to that name.

This dictionary creates that connection.

In [ ]:
AVAILABLE_TOOLS = {
    "calculator": calculator,
    "web_search": web_search,
    "get_weather": get_weather
}

print(AVAILABLE_TOOLS.keys())

## 14. Ask the LLM to Select a Tool

Let's send a request that clearly requires a calculator.

We are not executing the calculator ourselves.

We are asking the model:

> Which tool should I use?

In [ ]:
calculator_declaration = {
    "name": "calculator",
    "description": "Calculates a basic mathematical expression.",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A mathematical expression such as 25 * 18"
            }
        },
        "required": ["expression"]
    }
}


search_declaration = {
    "name": "web_search",
    "description": "Searches the web for current or specific information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query."
            }
        },
        "required": ["query"]
    }
}


weather_declaration = {
    "name": "get_weather",
    "description": "Gets the current weather for a city using an external weather API.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city."
            }
        },
        "required": ["city"]
    }
}


TOOL_DECLARATIONS = [
    calculator_declaration,
    search_declaration,
    weather_declaration
]

## 15. Execute the Tool

Now we take the function call returned by the model and execute the matching Python function.

This is the key part of the architecture:

```text
LLM
 ↓
Function Call
 ↓
Python
 ↓
Tool Result

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

MODEL = "gemini-3.8-flash"

In [ ]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Client created.")

In [ ]:
MODEL = "gemini-3.8-flash"

In [ ]:
print(MODEL)

Step 1 — Make sure your tools are defined

In [ ]:
TOOL_DECLARATIONS = [
    calculator_declaration,
    search_declaration,
    weather_declaration
]

AVAILABLE_TOOLS = {
    "calculator": calculator,
    "web_search": web_search,
    "get_weather": get_weather
}

print("Tools ready.")

# Step 2 — Ask Gemini to select a tool

In [ ]:
user_prompt = "What is 125 * 24?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_prompt,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )
)

print("Response received.")

# Step 3 — Check which tool Gemini selected

In [ ]:
function_call = None

for part in response.candidates[0].content.parts:
    if part.function_call:
        function_call = part.function_call
        break

if function_call:
    tool_name = function_call.name
    tool_args = dict(function_call.args)

    print("Tool:", tool_name)
    print("Arguments:", tool_args)
else:
    print("The model did not request a tool.")

# Step 4 — Execute the selected tool

In [ ]:
if function_call:
    tool_function = AVAILABLE_TOOLS[tool_name]

    tool_result = tool_function(**tool_args)

    print("Tool result:", tool_result)

# Step 5 — Send the tool result back to Gemini

In [ ]:
tool_response_part = types.Part.from_function_response(
    name=tool_name,
    response={
        "result": tool_result
    }
)

final_response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=user_prompt)
            ]
        ),
        response.candidates[0].content,
        types.Content(
            role="user",
            parts=[tool_response_part]
        )
    ],
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )
)

print(final_response.text)

# Step 6 — Build a reusable tool-calling agent



In [ ]:
def run_tool_agent(user_prompt):

    # 1. Ask Gemini to decide whether a tool is needed
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            tools=[
                types.Tool(
                    function_declarations=TOOL_DECLARATIONS
                )
            ]
        )
    )

    # 2. Find the function call
    function_call = None

    for part in response.candidates[0].content.parts:
        if part.function_call:
            function_call = part.function_call
            break

    # 3. If no tool is needed
    if not function_call:
        return response.text

    # 4. Get tool name and arguments
    tool_name = function_call.name
    tool_args = dict(function_call.args)

    print("Tool selected:", tool_name)
    print("Arguments:", tool_args)

    # 5. Execute the tool
    tool_function = AVAILABLE_TOOLS[tool_name]
    tool_result = tool_function(**tool_args)

    print("Tool result:", tool_result)

    # 6. Send the tool result back to Gemini
    tool_response_part = types.Part.from_function_response(
        name=tool_name,
        response={
            "result": tool_result
        }
    )

    final_response = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=user_prompt)
                ]
            ),
            response.candidates[0].content,
            types.Content(
                role="user",
                parts=[tool_response_part]
            )
        ],
        config=types.GenerateContentConfig(
            tools=[
                types.Tool(
                    function_declarations=TOOL_DECLARATIONS
                )
            ]
        )
    )

    return final_response.text

# Step 7 — Test the reusable agent

In [ ]:
answer = run_tool_agent("What is 25 * 18?")

print("\nFinal answer:")
print(answer)

# Build the Tool-Calling Workflow

In [ ]:
def run_tool_agent(user_prompt):

    # Step 1: Ask Gemini to choose a tool
    response = client.models.generate_content(
        model=MODEL,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            tools=[
                types.Tool(
                    function_declarations=TOOL_DECLARATIONS
                )
            ]
        )
    )

    # Step 2: Find the function call
    function_call = None

    for part in response.candidates[0].content.parts:
        if part.function_call:
            function_call = part.function_call
            break

    # Step 3: No tool needed
    if not function_call:
        return response.text

    # Step 4: Get tool information
    tool_name = function_call.name
    tool_args = dict(function_call.args)

    print("Tool selected:", tool_name)
    print("Arguments:", tool_args)

    # Step 5: Execute the tool
    tool_function = AVAILABLE_TOOLS[tool_name]
    tool_result = tool_function(**tool_args)

    print("Tool result:", tool_result)

    # Step 6: Ask Gemini to explain the tool result
    final_prompt = f"""
The user asked:

{user_prompt}

A tool was used to answer the request.

Tool used: {tool_name}

Tool result:
{tool_result}

Based on the tool result, give the user a clear and concise answer.
Do not call any tools. Use only the information provided in the tool result.
"""

    final_response = client.models.generate_content(
        model=MODEL,
        contents=final_prompt
    )

    return final_response.text

# Build a Complete Tool-Calling Agent

In [ ]:
def run_tool_agent(user_prompt):

    # Step 1: Start the conversation
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=user_prompt)
            ]
        )
    ]

    # Step 2: Give Gemini the available tools
    config = types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )

    # Step 3: Ask Gemini to decide whether a tool is needed
    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config
    )

    # Step 4: Check for a function call
    function_call = None

    for part in response.candidates[0].content.parts:
        if part.function_call:
            function_call = part.function_call
            break

    # Step 5: If no tool was requested, return Gemini's answer
    if not function_call:
        return response.text

    # Step 6: Get tool name and arguments
    tool_name = function_call.name
    tool_args = dict(function_call.args)

    print("Tool selected:", tool_name)
    print("Arguments:", tool_args)

    # Step 7: Execute the selected tool
    tool_function = AVAILABLE_TOOLS[tool_name]
    tool_result = tool_function(**tool_args)

    print("Tool result:", tool_result)

    # Step 8: Add Gemini's function-call response to the conversation
    contents.append(response.candidates[0].content)

    # Step 9: Create the tool response
    function_response_part = types.Part.from_function_response(
        name=tool_name,
        response={
            "result": tool_result
        }
    )

    # Step 10: Add the tool result to the conversation
    contents.append(
        types.Content(
            role="user",
            parts=[function_response_part]
        )
    )

    # Step 11: Ask Gemini for the final answer
    final_response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=config
    )

    return final_response.text

# Test the Web Search Tool

In [ ]:
search_result = web_search("LangGraph")

short_results = []

for item in search_result:
    short_results.append({
        "title": item.get("title"),
        "snippet": item.get("snippet", "")[:300]
    })

print(short_results)

# Add Retry Handling for Gemini API

In [ ]:
import time

def generate_with_retry(prompt, max_retries=4):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL,
                contents=prompt
            )

        except Exception as e:
            error_text = str(e)

            if "503" in error_text or "UNAVAILABLE" in error_text:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"Gemini temporarily unavailable. Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    raise

            else:
                raise

# Generate a Final Answer from Search Results

In [ ]:
final_prompt = f"""
Based on these web search results, explain LangGraph briefly:

{short_results}
"""

test_final = generate_with_retry(final_prompt)

print(test_final.text)

# Improve the Gemini Retry Function

In [ ]:
import time

def generate_with_retry(prompt, config=None, max_retries=4):

    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=config
            )

        except Exception as e:

            error_text = str(e)

            if "503" in error_text or "UNAVAILABLE" in error_text:

                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt

                    print(
                        f"Gemini temporarily unavailable. "
                        f"Retrying in {wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                else:
                    raise

            else:
                raise

# Build the Final Tool-Calling Agent

In [ ]:
def run_tool_agent(user_prompt):

    # Step 1: Give Gemini the available tools
    config = types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )

    # Step 2: Ask Gemini to choose a tool
    response = generate_with_retry(
        user_prompt,
        config=config
    )

    # Step 3: Find the function call
    function_call = None

    for part in response.candidates[0].content.parts:
        if part.function_call:
            function_call = part.function_call
            break

    # Step 4: No tool needed
    if not function_call:
        return response.text

    # Step 5: Get tool information
    tool_name = function_call.name
    tool_args = dict(function_call.args)

    print("Tool selected:", tool_name)
    print("Arguments:", tool_args)

    # Step 6: Execute the tool
    tool_function = AVAILABLE_TOOLS[tool_name]
    tool_result = tool_function(**tool_args)

    print("Tool result:", tool_result)

    # Step 7: Keep search results short
    if isinstance(tool_result, list):

        short_results = []

        for item in tool_result:
            short_results.append({
                "title": item.get("title"),
                "snippet": item.get("snippet", "")[:300]
            })

        tool_result = short_results

    # Step 8: Prepare final prompt
    final_prompt = f"""
User request:
{user_prompt}

Tool used:
{tool_name}

Tool result:
{tool_result}

Answer the user's request using the tool result.
Do not use any tools.
"""

    # Step 9: Ask Gemini for the final answer
    final_response = generate_with_retry(
        final_prompt
    )

    return final_response.text

# Test the Tool-Calling Agent

In [ ]:
answer = run_tool_agent(
    "Search the web for information about LangGraph."
)

print("\nFinal answer:")
print(answer)